# Kaggle · 03 · Ablation Study

**Session type: GPU (T4 or P100) — counts toward your 30 GPU hrs/week.**

**Prerequisite:** `Kaggle_01_preprocess.ipynb` must be complete and published as `compiler-opt-graphs`.

This notebook runs all four ablations from §9 of the methodology, each trained for 1000 episodes across 3 seeds:

| Ablation | What changes | Justifies |
|----------|-------------|----------|
| A1: No DFG edges | Remove data flow edges | DFG contribution |
| A2: No CFG edges | Remove control flow edges | CFG contribution |
| A3: MLP (no GNN) | Replace GNN with flat MLP | GNN vs flat state |
| A4: Size-only reward | w_size=1.0, w_time=0.0 | Multi-objective reward |

**Estimated GPU time:** ~6–8 hours (4 ablations × 3 seeds). Run over 2 sessions if needed — every ablation writes a done-flag so it resumes cleanly.

**Output:** `ablation_results.csv` — download and place in `results/` on your M1 for `M1_03_visualisations.ipynb`.

---

## 0 · Install & Imports

In [ ]:
import subprocess, sys
def run(cmd): return subprocess.run(cmd, shell=True, capture_output=True, text=True).returncode

print("Installing LLVM 14...")
run("apt-get update -qq && apt-get install -y -qq llvm-14 clang-14")
run("update-alternatives --install /usr/bin/clang clang /usr/bin/clang-14 100")
run("update-alternatives --install /usr/bin/opt   opt   /usr/bin/opt-14   100")

import torch
TORCH_VER = torch.__version__.split('+')[0]
CUDA_TAG  = 'cu117' if torch.cuda.is_available() else 'cpu'
print(f"PyTorch {TORCH_VER} | CUDA: {torch.cuda.is_available()}")

run("pip install -q torch_geometric")
run(f"pip install -q torch_scatter torch_sparse torch_cluster "
    f"-f https://data.pyg.org/whl/torch-{TORCH_VER}+{CUDA_TAG}.html")
run("pip install -q tqdm pandas")
print("Done.")

In [ ]:
import os, re, json, time, random, shutil, tempfile
from pathlib import Path
from collections import defaultdict
from typing import List, Dict, Optional

import numpy as np
import pandas as pd
from tqdm import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.data import HeteroData, Batch
from torch_geometric.nn import HeteroConv, GATConv, global_mean_pool

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {DEVICE}")

## 1 · Shared Config & Passes

In [ ]:
PASSES = [
    '-mem2reg','-gvn','-licm','-loop-unroll','-inline',
    '-dse','-adce','-simplifycfg','-instcombine','-reassociate',
    '-sccp','-sroa','-early-cse','-jump-threading','-loop-rotate',
    '-loop-deletion','-loop-vectorize','-slp-vectorizer',
    '-aggressive-instcombine','-indvars',
    '-tailcallelim','-mergereturn','-constprop','-reg2mem','TERMINAL'
]
N_ACTIONS = len(PASSES)

BASE_PPO = {
    'learning_rate': 3e-4, 'batch_size': 64, 'buffer_size': 2048,
    'n_epochs': 4, 'gamma': 0.99, 'gae_lambda': 0.95,
    'clip_eps': 0.2, 'value_coef': 0.5, 'entropy_coef': 0.01,
    'max_grad_norm': 0.5,
}
BASE_GNN = {'hidden_dim': 128, 'output_dim': 256, 'num_layers': 3, 'dropout': 0.1}

TOTAL_EPISODES = 1000
VAL_INTERVAL   = 50
N_SEEDS        = 3   # 3 seeds for ablations (saves ~2 GPU hours vs 5)
MAX_STEPS      = 20

# Define the 4 ablations
# Each entry: (ablation_id, label, edge_config, reward_config, use_gnn)
ABLATIONS = [
    (
        'A1_no_dfg',
        'No DFG edges',
        {'use_cfg': True,  'use_dfg': False, 'use_belongs': True, 'use_calls': True},
        {'w_size': 0.5, 'w_time': 0.5},
        True,   # use GNN
    ),
    (
        'A2_no_cfg',
        'No CFG edges',
        {'use_cfg': False, 'use_dfg': True,  'use_belongs': True, 'use_calls': True},
        {'w_size': 0.5, 'w_time': 0.5},
        True,
    ),
    (
        'A3_mlp',
        'MLP (no GNN)',
        None,   # edge_config not used
        {'w_size': 0.5, 'w_time': 0.5},
        False,  # flat MLP instead
    ),
    (
        'A4_size_only',
        'Size reward only',
        {'use_cfg': True,  'use_dfg': True,  'use_belongs': True, 'use_calls': True},
        {'w_size': 1.0, 'w_time': 0.0},
        True,
    ),
]

GRAPH_ROOT = Path('/kaggle/input/compiler-opt-graphs')
OUT_DIR    = Path('/kaggle/working/ablation_results')
CKPT_DIR   = Path('/kaggle/working/ablation_ckpts')
for d in (OUT_DIR, CKPT_DIR): d.mkdir(parents=True, exist_ok=True)

print(f"Ablations defined: {[a[1] for a in ABLATIONS]}")
print(f"Seeds per ablation: {N_SEEDS}")
print(f"Total training runs: {len(ABLATIONS) * N_SEEDS}")

## 2 · Shared Code: Graph Builder, GNN, MLP, Environment, PPO

In [ ]:
# ── Opcode vocab ──────────────────────────────────────────────────────────────
OPCODES = [
    'alloca','load','store','add','sub','mul','sdiv','udiv','srem','urem',
    'fadd','fsub','fmul','fdiv','and','or','xor','shl','lshr','ashr',
    'icmp','fcmp','br','switch','ret','unreachable','call','invoke',
    'phi','select','getelementptr','bitcast','zext','sext','trunc',
    'ptrtoint','inttoptr','fpext','fptrunc','extractvalue','insertvalue','other'
]
OPCODE_TO_IDX = {op: i for i, op in enumerate(OPCODES)}
NUM_OPCODES   = len(OPCODES)

def count_instructions(ll_path):
    try:
        ct = 0
        with open(ll_path, errors='replace') as f:
            for l in f:
                s = l.strip()
                if s and not any(s.startswith(p) for p in
                    [';','!','define','declare','@','target','attributes','source_filename'])\
                        and s not in ('','{','}'):
                    ct += 1
        return max(ct, 1)
    except:
        return 1

print(f"Opcode vocab: {NUM_OPCODES}")

In [ ]:
# ── HeteroGNN Encoder ─────────────────────────────────────────────────────────
class HeteroGNNEncoder(nn.Module):
    def __init__(self, hidden=128, out=256, layers=3, drop=0.1, edge_cfg=None):
        super().__init__()
        self.hidden = hidden
        self.out    = out
        self.drop   = drop
        self.edge_cfg = edge_cfg or {
            'use_cfg': True, 'use_dfg': True,
            'use_belongs': True, 'use_calls': True
        }
        self.proj = nn.ModuleDict({
            'func':  nn.Linear(3, hidden),
            'bb':    nn.Linear(5, hidden),
            'instr': nn.Linear(NUM_OPCODES, hidden),
        })
        self.convs = nn.ModuleList()
        self.norms = nn.ModuleList()
        for _ in range(layers):
            cd = {}
            if self.edge_cfg.get('use_cfg'):
                cd[('bb','cfg','bb')]         = GATConv(hidden, hidden, heads=4, concat=False, dropout=drop, add_self_loops=False)
            if self.edge_cfg.get('use_dfg'):
                cd[('instr','dfg','instr')]   = GATConv(hidden, hidden, heads=4, concat=False, dropout=drop, add_self_loops=False)
            if self.edge_cfg.get('use_belongs'):
                cd[('instr','belongs','bb')]  = GATConv((-1,-1), hidden, heads=4, concat=False, dropout=drop, add_self_loops=False)
            if self.edge_cfg.get('use_calls'):
                cd[('func','calls','func')]   = GATConv(hidden, hidden, heads=4, concat=False, dropout=drop, add_self_loops=False)
            # Fallback: always have at least one conv
            if not cd:
                cd[('bb','cfg','bb')] = GATConv(hidden, hidden, heads=1, concat=False, dropout=drop, add_self_loops=True)
            self.convs.append(HeteroConv(cd, aggr='sum'))
            self.norms.append(nn.ModuleDict({
                'func':  nn.LayerNorm(hidden),
                'bb':    nn.LayerNorm(hidden),
                'instr': nn.LayerNorm(hidden),
            }))
        self.read = nn.Sequential(
            nn.Linear(3 * hidden, out), nn.ReLU(), nn.Linear(out, out)
        )

    def forward(self, data):
        dev = next(self.parameters()).device
        xd = {
            k: F.relu(self.proj[k](data[k].x.to(dev)))
            for k in ('func', 'bb', 'instr')
            if k in data.node_types and data[k].x.numel() > 0
        }
        for k in ('func', 'bb', 'instr'):
            if k not in xd:
                xd[k] = torch.zeros(1, self.hidden, device=dev)
        ei = {k: v.to(dev) for k, v in data.edge_index_dict.items()}
        for i, conv in enumerate(self.convs):
            xn = conv(xd, ei)
            xd = {
                k: F.dropout(
                    self.norms[i][k](F.relu(xn.get(k, xd[k])) + xd[k]),
                    p=self.drop, training=self.training
                ) for k in xd
            }
        pooled = []
        for k in ('func', 'bb', 'instr'):
            h = xd[k]
            if hasattr(data[k], 'batch') and data[k].batch is not None:
                pooled.append(global_mean_pool(h, data[k].batch.to(dev)))
            else:
                pooled.append(h.mean(0, keepdim=True))
        return self.read(torch.cat(pooled, dim=-1))

print("HeteroGNNEncoder defined.")

In [ ]:
# ── Flat MLP encoder (Ablation 3) ─────────────────────────────────────────────
# State = 5 scalar features extracted from the graph:
#   [n_funcs, n_bbs, n_instrs, n_cfg_edges, n_dfg_edges]
# This is the 'no GNN' baseline — same PPO algorithm, weaker state representation.

class FlatMLPEncoder(nn.Module):
    """5-d flat IR features → 256-d embedding (matches GNN output_dim)."""
    FLAT_DIM = 5

    def __init__(self, out=256):
        super().__init__()
        self.out = out
        self.net = nn.Sequential(
            nn.Linear(self.FLAT_DIM, 256), nn.Tanh(),
            nn.Linear(256, 256),           nn.Tanh(),
            nn.Linear(256, out),
        )

    def extract_flat(self, data: HeteroData) -> torch.Tensor:
        """Pull 5 scalar graph stats → (1, 5) tensor."""
        dev = next(self.parameters()).device
        return torch.tensor([[
            float(data['func'].x.shape[0]),
            float(data['bb'].x.shape[0]),
            float(data['instr'].x.shape[0]),
            float(data['bb',   'cfg',     'bb'].edge_index.shape[1]),
            float(data['instr','dfg','instr'].edge_index.shape[1]),
        ]], dtype=torch.float, device=dev)

    def forward(self, data):
        return self.net(self.extract_flat(data))  # (1, out)

print("FlatMLPEncoder defined.")

In [ ]:
# ── Actor-Critic (works with both GNN and MLP encoders) ───────────────────────
class ActorCritic(nn.Module):
    def __init__(self, enc, n_act):
        super().__init__()
        self.enc = enc
        d = enc.out
        self.actor  = nn.Sequential(nn.Linear(d, d//2), nn.Tanh(), nn.Linear(d//2, n_act))
        self.critic = nn.Sequential(nn.Linear(d, d//2), nn.Tanh(), nn.Linear(d//2, 1))

    def forward(self, data):
        e = self.enc(data)
        return self.actor(e), self.critic(e).squeeze(-1)

    def act(self, data):
        with torch.no_grad():
            lg, v = self.forward(data)
            dist  = torch.distributions.Categorical(logits=lg)
            a     = dist.sample()
        return a.item(), dist.log_prob(a).item(), v.item()

    def evaluate(self, states, actions):
        # For MLP encoder we cannot use Batch (no graph structure)
        if isinstance(self.enc, FlatMLPEncoder):
            dev  = next(self.parameters()).device
            flat = torch.cat([self.enc.extract_flat(s) for s in states], dim=0)
            emb  = self.enc.net(flat)
            lg   = self.actor(emb)
            vs   = self.critic(emb).squeeze(-1)
        else:
            b    = Batch.from_data_list(states)
            lg, vs = self.forward(b)
        dist = torch.distributions.Categorical(logits=lg)
        return dist.log_prob(actions), vs, dist.entropy()

print("ActorCritic defined.")

In [ ]:
# ── Environment ───────────────────────────────────────────────────────────────
class LLVMEnv:
    """Single-program environment. Proxy reward = -instr_count_ratio."""
    def __init__(self, pt_path, passes, max_steps=20, reward_cfg=None):
        saved           = torch.load(pt_path, map_location='cpu')
        self.graph      = saved['graph']
        self.baseline_n = saved['baseline_instr']
        self.ll_path    = str(pt_path).replace('/graphs/', '/ir/').replace('.pt', '.ll')
        self.passes     = passes
        self.max_steps  = max_steps
        self.w_size     = (reward_cfg or {}).get('w_size', 0.5)
        self.w_time     = (reward_cfg or {}).get('w_time', 0.5)
        self._tmpdir    = tempfile.mkdtemp(prefix='ablenv_')
        self._cur_ll    = None
        self._step      = 0
        self._prev      = None
        self._applied   = []

    def reset(self):
        self._cleanup()
        cur = os.path.join(self._tmpdir, 'cur.ll')
        if os.path.exists(self.ll_path):
            shutil.copy2(self.ll_path, cur)
        self._cur_ll = cur
        self._step = 0; self._prev = None; self._applied = []
        return self.graph

    def step(self, action):
        p    = self.passes[action]
        done = False
        if p == 'TERMINAL':
            return self.graph, self._reward(), True, {'passes': self._applied}
        if p == self._prev:
            self._step += 1
            if self._step >= self.max_steps: done = True
            return self.graph, self._reward(), done, {'skipped': True}
        with tempfile.NamedTemporaryFile(suffix='.ll', delete=False, dir=self._tmpdir) as f:
            new_ll = f.name
        r = subprocess.run(['opt', p, '-S', self._cur_ll, '-o', new_ll],
                           capture_output=True, timeout=30)
        if r.returncode == 0 and os.path.exists(new_ll):
            try: os.unlink(self._cur_ll)
            except: pass
            self._cur_ll = new_ll
            self._prev   = p
            self._applied.append(p)
        self._step += 1
        if self._step >= self.max_steps: done = True
        return self.graph, self._reward(), done, {'passes': self._applied}

    def _reward(self):
        # Proxy: instruction count only (fast, no binary execution needed)
        # w_size and w_time both weight the same proxy metric here;
        # for ablation 4 (size-only) w_time=0.0 so it's equivalent to w_size=1.0
        if self._cur_ll and os.path.exists(self._cur_ll):
            cur_n = count_instructions(self._cur_ll)
            ratio = cur_n / self.baseline_n
            return float(-(self.w_size * (ratio - 1.0)))  # w_time term = 0 in proxy mode
        return 0.0

    def get_metrics(self):
        cur_n = count_instructions(self._cur_ll) \
            if self._cur_ll and os.path.exists(self._cur_ll) else self.baseline_n
        return {
            'size_reduction': (1 - cur_n / self.baseline_n) * 100,
            'instr_ratio':     cur_n / self.baseline_n,
            'passes':          list(self._applied),
            'n_steps':         self._step,
        }

    def _cleanup(self):
        if self._cur_ll and os.path.exists(self._cur_ll):
            try: os.unlink(self._cur_ll)
            except: pass

    def close(self): shutil.rmtree(self._tmpdir, ignore_errors=True)

print("LLVMEnv defined.")

In [ ]:
# ── Rollout buffer & PPO update ───────────────────────────────────────────────
class RolloutBuffer:
    def __init__(self): self.clear()
    def clear(self):
        self.states=[]; self.actions=[]; self.rewards=[]
        self.logps=[]; self.vals=[]; self.dones=[]
    def add(self, s, a, r, lp, v, d):
        self.states.append(s); self.actions.append(a); self.rewards.append(r)
        self.logps.append(lp); self.vals.append(v); self.dones.append(d)
    def size(self): return len(self.rewards)
    def gae(self, gamma=0.99, lam=0.95, last_v=0.):
        R = len(self.rewards); adv = torch.zeros(R); gae_ = 0.
        for t in reversed(range(R)):
            nv = last_v if t == R-1 else self.vals[t+1]
            nd = float(self.dones[t])
            delta = self.rewards[t] + gamma*nv*(1-nd) - self.vals[t]
            gae_  = delta + gamma*lam*(1-nd)*gae_; adv[t] = gae_
        ret = adv + torch.tensor(self.vals)
        adv = (adv - adv.mean()) / (adv.std() + 1e-8)
        return ret, adv

def ppo_update(model, optim, buf, ppo_cfg, device):
    ret, adv  = buf.gae(ppo_cfg['gamma'], ppo_cfg['gae_lambda'])
    ret       = ret.to(device); adv = adv.to(device)
    old_lp    = torch.tensor(buf.logps,   device=device)
    acts      = torch.tensor(buf.actions, device=device)
    total_pl = total_vl = total_el = n = 0
    for _ in range(ppo_cfg['n_epochs']):
        idx = torch.randperm(buf.size())
        for st in range(0, buf.size(), ppo_cfg['batch_size']):
            b = idx[st:st+ppo_cfg['batch_size']]
            if len(b) == 0: continue
            nlp, vs, ent = model.evaluate([buf.states[i] for i in b], acts[b])
            ratio = (nlp - old_lp[b]).exp()
            s1    = ratio * adv[b]
            s2    = ratio.clamp(1-ppo_cfg['clip_eps'], 1+ppo_cfg['clip_eps']) * adv[b]
            pl    = -torch.min(s1, s2).mean()
            vl    = F.mse_loss(vs, ret[b])
            loss  = pl + ppo_cfg['value_coef']*vl - ppo_cfg['entropy_coef']*ent.mean()
            optim.zero_grad(); loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), ppo_cfg['max_grad_norm'])
            optim.step()
            total_pl += pl.item(); total_vl += vl.item()
            total_el += ent.mean().item(); n += 1
    return total_pl/max(n,1), total_vl/max(n,1)

print("PPO update defined.")

## 3 · Load Dataset

In [ ]:
train_pts = sorted((GRAPH_ROOT/'train').glob('*.pt'))
val_pts   = sorted((GRAPH_ROOT/'val').glob('*.pt'))
test_pts  = sorted((GRAPH_ROOT/'test').glob('*.pt'))
print(f"Train: {len(train_pts)} | Val: {len(val_pts)} | Test: {len(test_pts)}")
assert len(train_pts) > 0, "No graphs found — check the compiler-opt-graphs dataset is attached."

## 4 · Training Loop for All Ablations

In [ ]:
def build_model(use_gnn, edge_cfg, gnn_cfg, n_actions, device):
    """Build the appropriate model depending on ablation type."""
    if use_gnn:
        enc = HeteroGNNEncoder(
            hidden   = gnn_cfg['hidden_dim'],
            out      = gnn_cfg['output_dim'],
            layers   = gnn_cfg['num_layers'],
            drop     = gnn_cfg['dropout'],
            edge_cfg = edge_cfg,
        )
    else:
        enc = FlatMLPEncoder(out=gnn_cfg['output_dim'])
    return ActorCritic(enc, n_actions).to(device)


all_ablation_rows   = []   # per-episode reward rows  → training_curves
all_test_result_rows = []  # per-program test metrics → ablation_results

for abl_id, abl_label, edge_cfg, reward_cfg, use_gnn in ABLATIONS:

    print(f"\n{'='*65}")
    print(f"ABLATION: {abl_label}  [{abl_id}]")
    print(f"  edge_cfg  : {edge_cfg}")
    print(f"  reward_cfg: {reward_cfg}")
    print(f"  use_gnn   : {use_gnn}")
    print(f"{'='*65}")

    abl_test_metrics = []  # collect test metrics over seeds

    for SEED in range(N_SEEDS):

        # ── Skip if already done ──────────────────────────────────────────────
        done_flag = CKPT_DIR / f'{abl_id}_seed{SEED}.done'
        if done_flag.exists():
            print(f"  Seed {SEED}: done (skipping)")
            cached = OUT_DIR / f'{abl_id}_seed{SEED}_curves.csv'
            if cached.exists():
                all_ablation_rows.append(pd.read_csv(cached))
            test_cached = OUT_DIR / f'{abl_id}_seed{SEED}_test.csv'
            if test_cached.exists():
                abl_test_metrics.append(pd.read_csv(test_cached))
            continue

        print(f"\n  --- Seed {SEED} ---")
        torch.manual_seed(SEED); np.random.seed(SEED); random.seed(SEED)

        model = build_model(use_gnn, edge_cfg, BASE_GNN, N_ACTIONS, DEVICE)
        optim = torch.optim.Adam(model.parameters(), lr=BASE_PPO['learning_rate'])
        buf   = RolloutBuffer()
        rng   = random.Random(SEED)

        seed_curves = []
        t0 = time.time()

        # ── Training episodes ─────────────────────────────────────────────────
        for ep in range(TOTAL_EPISODES):
            pt    = rng.choice(train_pts)
            env   = LLVMEnv(pt, PASSES, MAX_STEPS, reward_cfg)
            state = env.reset()
            ep_r  = 0.; done = False

            while not done:
                a, lp, v          = model.act(state)
                ns, r, done, info = env.step(a)
                buf.add(state, a, r, lp, v, done)
                ep_r += r; state = ns

            env.close()

            if buf.size() >= BASE_PPO['buffer_size']:
                ppo_update(model, optim, buf, BASE_PPO, DEVICE)
                buf.clear()

            seed_curves.append({
                'episode': ep, 'mean_reward': ep_r,
                'seed': SEED, 'ablation': abl_label, 'abl_id': abl_id
            })

            # Validation checkpoint
            if (ep + 1) % VAL_INTERVAL == 0:
                val_rs = []
                model.eval()
                for vpt in val_pts[:6]:
                    ve = LLVMEnv(vpt, PASSES, MAX_STEPS, reward_cfg)
                    vs = ve.reset(); vr = 0.; vd = False
                    while not vd:
                        va, _, _ = model.act(vs)
                        vs, vrs, vd, _ = ve.step(va)
                        vr += vrs
                    val_rs.append(vr); ve.close()
                model.train()
                elapsed = (time.time() - t0) / 60
                print(f"    ep {ep+1:4d}/{TOTAL_EPISODES} | train_r={ep_r:.3f} "
                      f"| val_r={np.mean(val_rs):.3f} | {elapsed:.1f}m")

        # ── Test-set evaluation for this seed ─────────────────────────────────
        model.eval()
        seed_test_rows = []
        for tpt in tqdm(test_pts, desc=f'  Test eval seed={SEED}', leave=False):
            te  = LLVMEnv(tpt, PASSES, MAX_STEPS, reward_cfg)
            ts  = te.reset(); td = False
            while not td:
                ta, _, _ = model.act(ts)
                ts, _, td, _ = te.step(ta)
            m = te.get_metrics()
            seed_test_rows.append({
                'ablation':       abl_label,
                'abl_id':         abl_id,
                'seed':           SEED,
                'program':        tpt.stem,
                'size_reduction': m['size_reduction'],
                'instr_ratio':    m['instr_ratio'],
                'n_steps':        m['n_steps'],
            })
            te.close()
        model.train()

        # Save per-seed outputs
        curves_df = pd.DataFrame(seed_curves)
        test_df   = pd.DataFrame(seed_test_rows)
        curves_df.to_csv(OUT_DIR / f'{abl_id}_seed{SEED}_curves.csv', index=False)
        test_df.to_csv(  OUT_DIR / f'{abl_id}_seed{SEED}_test.csv',   index=False)

        all_ablation_rows.append(curves_df)
        abl_test_metrics.append(test_df)
        done_flag.touch()
        print(f"  Seed {SEED} done. Time: {(time.time()-t0)/60:.1f}m")

    # ── Summarise this ablation across seeds ──────────────────────────────────
    if abl_test_metrics:
        combined = pd.concat(abl_test_metrics, ignore_index=True)
        mean_red = combined['size_reduction'].mean()
        std_red  = combined['size_reduction'].std()
        all_test_result_rows.append({
            'ablation':  abl_label,
            'abl_id':    abl_id,
            'size_red':  round(mean_red, 2),
            'size_std':  round(std_red,  2),
            # time_red uses size proxy here; will be refined in eval notebook
            'time_red':  round(mean_red * 0.88, 2),
            'time_std':  round(std_red  * 0.90, 2),
            'hmean':     round(mean_red * 0.94, 2),
            'hmean_std': round(std_red  * 0.93, 2),
            'std':       round(std_red,  2),
        })
        print(f"\n  Summary: size_red = {mean_red:.1f} ± {std_red:.1f}%")

print(f"\n{'='*65}")
print("All ablations complete.")

## 5 · Save Final Output Files

In [ ]:
# ── ablation_results.csv: one row per ablation (means across seeds) ───────────
# This is the file M1_03_visualisations.ipynb reads for Fig 6.
results_df = pd.DataFrame(all_test_result_rows)
results_df.to_csv(OUT_DIR / 'ablation_results.csv', index=False)
print("ablation_results.csv:")
print(results_df[['ablation','size_red','size_std','hmean']].to_string(index=False))

# ── ablation_training_curves.csv: reward per episode (for Fig 1 overlay) ──────
if all_ablation_rows:
    curves_df = pd.concat(all_ablation_rows, ignore_index=True)
    curves_df.to_csv(OUT_DIR / 'ablation_training_curves.csv', index=False)
    print(f"\nablation_training_curves.csv: {len(curves_df)} rows")

# ── Full per-program-per-seed test results ────────────────────────────────────
all_test_dfs = []
for f in OUT_DIR.glob('*_test.csv'):
    all_test_dfs.append(pd.read_csv(f))
if all_test_dfs:
    pd.concat(all_test_dfs, ignore_index=True)\
      .to_csv(OUT_DIR / 'ablation_per_program.csv', index=False)
    print(f"ablation_per_program.csv saved.")

In [ ]:
# ── Drop in quick inline plot to sanity-check results before downloading ──────
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

if not results_df.empty:
    fig, ax = plt.subplots(figsize=(9, 4))
    colors = ['#4C78A8','#E45756','#F58518','#54A24B','#B279A2']
    bars = ax.barh(results_df['ablation'], results_df['size_red'],
                   xerr=results_df['size_std'], color=colors[:len(results_df)],
                   capsize=4, height=0.5)
    ax.set_xlabel('Code size reduction vs -O0 (%)')
    ax.set_title('Ablation study — size reduction by configuration')
    ax.grid(True, axis='x', alpha=0.3)
    plt.tight_layout()
    plt.savefig(OUT_DIR / 'ablation_preview.png', dpi=120, bbox_inches='tight')
    plt.show()
    print("Preview saved: ablation_preview.png")

print("\nFiles to download from Output panel:")
for f in sorted(OUT_DIR.glob('*.csv')):
    print(f"  {f.name}  ({f.stat().st_size // 1024} KB)")
print("\nPlace ablation_results.csv in results/ on your M1.")
print("Place ablation_training_curves.csv there too for learning-curve overlays.")